# 05 · Overfitting, regularisation and cross-validation

After this notebook you can diagnose under- and overfitting from train-vs-validation curves, explain bias and variance with a real decomposition, read a learning curve to decide whether more data will help, use Ridge and Lasso (and say why L1 gives zeros), apply early stopping, and tune hyperparameters with cross-validation without fooling yourself.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [02 · Linear regression](02_linear_regression_and_gradient_descent.ipynb), [04 · Metrics](04_metrics_confusion_matrix_roc_pr.ipynb)

## Why this matters in interviews

- "It does great on training data and badly in production. What is happening?" is the canonical overfitting question. For LLM fine-tuning it is the same story: training loss falls while eval loss rises.
- "How did you pick your hyperparameters?" probes whether you know cross-validation and keep an untouched test set. The LLM version: tuning a prompt against the same 50 eval cases for six weeks.
- Regularisation (L1, L2, early stopping, dropout, weight decay) is one family of fixes. Knowing which to reach for, and why, signals real experience.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tn28` | What is overfitting in the context of fine-tuning an LLM, and how do you spot it? |
| `ev26` | Your offline evals improved and production got worse. What happened? |
| `ev15` | What is a golden dataset and how do you keep it useful? |
| `tn20` | How do you evaluate a fine-tuned model before shipping it? |
| `tn13` | How much data do you need to fine-tune, and what does good data look like? |
| `fo06` | What are parameters, and does a bigger parameter count mean a better model? |

**Also asked in classical-ML rounds:** explain the bias–variance trade-off; L1 vs L2 and why L1 is sparse; how do you read a learning curve? k-fold vs stratified vs group vs time-series splits; what is nested cross-validation for?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import load_breast_cancer, load_diabetes, load_digits
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge, RidgeCV, lasso_path
from sklearn.model_selection import (GridSearchCV, KFold, RepeatedStratifiedKFold, StratifiedKFold, TimeSeriesSplit,
                                     cross_val_score, learning_curve, train_test_split, validation_curve)
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

rng = np.random.default_rng(0)
BLUE, ORANGE, AQUA, YELLOW, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a8984"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA, YELLOW, "#4a3aa7"])})

## 1. Underfitting and overfitting: polynomial degrees 1, 4 and 15

**In plain English:** a model that is too simple misses the pattern (**underfits**: bad on train *and* validation). A model that is too flexible memorises the noise in its training points (**overfits**: great on train, bad on anything new). Twenty noisy samples of a smooth curve, fit with polynomials of degree 1, 4 and 15:

In [ ]:
true_f = lambda x: np.cos(1.5 * np.pi * x)
n = 20
x = np.sort(rng.random(n))
y = true_f(x) + rng.normal(0, 0.2, n)                 # the training set
x_val = rng.random(2000)
y_val = true_f(x_val) + rng.normal(0, 0.2, 2000)      # lots of fresh data from the same process

def poly_model(degree):
    return make_pipeline(PolynomialFeatures(degree, include_bias=False), LinearRegression())

mse = lambda m, xs, ys: np.mean((m.predict(xs[:, None]) - ys) ** 2)
grid = np.linspace(0, 1, 300)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), sharey=True)
results = {}
for ax, d in zip(axes, [1, 4, 15]):
    m = poly_model(d).fit(x[:, None], y)
    results[d] = (mse(m, x, y), mse(m, x_val, y_val))
    ax.plot(grid, true_f(grid), color=GREY, ls="--", label="true function")
    ax.plot(grid, m.predict(grid[:, None]), color=ORANGE, lw=2, label=f"degree {d} fit")
    ax.scatter(x, y, s=18, color=BLUE, zorder=3, label="training points")
    ax.set(title=f"degree {d}\ntrain MSE {results[d][0]:.3f} | val MSE {results[d][1]:.3f}", xlabel="x", ylim=(-2, 2))
axes[0].set_ylabel("y"); axes[0].legend(loc="lower left", fontsize=8)
plt.tight_layout(); plt.show()
assert results[15][0] < results[4][0] < results[1][0]          # train error only goes down with capacity
assert results[4][1] < results[1][1] and results[4][1] < results[15][1] / 10

Degree 1 underfits (both errors high). Degree 4 tracks the curve. Degree 15 has the *lowest* training error of the three and a validation error about fifteen times worse: where the training points run out (look at x near 1) it swings wildly. More parameters bought memorisation, not skill (`fo06`).

## 2. Train vs validation error as capacity grows

Sweep the degree and plot both errors. Training error can only fall; validation error falls, bottoms out, then climbs. The bottom is the model you want.

In [ ]:
degrees = np.arange(1, 16)
train_err, val_err = [], []
for d in degrees:
    m = poly_model(d).fit(x[:, None], y)
    train_err.append(mse(m, x, y)); val_err.append(mse(m, x_val, y_val))
best_d = degrees[int(np.argmin(val_err))]

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(degrees, train_err, "o-", lw=2, label="training error")
ax.plot(degrees, val_err, "o-", lw=2, label="validation error")
ax.axvline(best_d, color=GREY, ls="--", label=f"best degree = {best_d}")
ax.axhline(0.2 ** 2, color=GREY, lw=1, ls=":", label="noise floor (0.04)")
ax.set_yscale("log")
ax.set(title="Error vs model capacity", xlabel="polynomial degree (capacity)", ylabel="MSE (log)")
ax.text(1.3, 0.3, "underfitting", fontsize=9); ax.text(11, 0.3, "overfitting", fontsize=9)
ax.legend(loc="upper center"); plt.show()
print(f"best degree on validation: {best_d}")
assert 3 <= best_d <= 6 and val_err[-1] > 10 * min(val_err)

Note the dotted line: no model can beat the noise in the data (MSE 0.04 here). A training error *below* it is a sign you are fitting noise.

## 3. Bias and variance, measured

**In plain English:** imagine re-collecting the training set many times. **Bias** is how far the *average* fitted curve is from the truth (a systematic miss). **Variance** is how much the fitted curve jumps around from one training set to the next. Expected error = bias² + variance + irreducible noise.

We can measure both directly: draw 200 fresh training sets of 20 points, fit each degree, and look at the spread of predictions.

In [ ]:
reps, eval_x = 200, np.linspace(0.05, 0.95, 100)        # stay inside the data range
rng_bv = np.random.default_rng(1)
bias2, variance = [], []
deg_bv = np.arange(1, 8)
for d in deg_bv:
    preds = np.empty((reps, eval_x.size))
    for r in range(reps):
        xs = rng_bv.random(n)
        ys = true_f(xs) + rng_bv.normal(0, 0.2, n)
        preds[r] = np.polynomial.Polynomial.fit(xs, ys, d)(eval_x)   # least-squares polynomial fit
    bias2.append(np.mean((preds.mean(axis=0) - true_f(eval_x)) ** 2))
    variance.append(np.mean(preds.var(axis=0)))
total = np.array(bias2) + np.array(variance) + 0.2 ** 2

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(deg_bv, bias2, "o-", lw=2, label="bias²")
ax.plot(deg_bv, variance, "o-", lw=2, label="variance")
ax.plot(deg_bv, total, "o-", lw=2.5, color="black", label="bias² + variance + noise")
ax.set_yscale("log")
ax.set(title="Bias-variance decomposition over 200 training sets", xlabel="polynomial degree", ylabel="error (log)")
ax.legend(); plt.show()
print(f"lowest expected error at degree {deg_bv[np.argmin(total)]}")
assert bias2[0] > 100 * min(bias2) and variance[-1] > 20 * min(variance)
assert 2 <= deg_bv[np.argmin(total)] <= 5

| | High bias (underfit) | High variance (overfit) |
|---|---|---|
| Symptom | train and validation errors both high, close together | train error low, validation error much higher |
| Typical cause | model too simple, missing features, too much regularisation | model too flexible, too little data, too little regularisation |
| Fixes | richer model or features, less regularisation, train longer | more data, regularisation, simpler model, early stopping, ensembling (bagging) |

## 4. Learning curves: will more data help? (`tn13`)

A **learning curve** plots training and validation scores against the *number of training examples*. It answers the most expensive question in ML, "should we label more data?", before you pay for it. Two models on the digits dataset (8×8 images of handwritten digits):

In [ ]:
X_dig, y_dig = load_digits(return_X_y=True)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
curves = {}
for ax, (name, est) in zip(axes, [("Gaussian naive Bayes (simple)", GaussianNB()),
                                  ("decision tree, no depth limit (flexible)", DecisionTreeClassifier(random_state=0))]):
    sizes, tr_sc, va_sc = learning_curve(est, X_dig, y_dig, cv=5, train_sizes=np.linspace(0.1, 1.0, 6))
    curves[name] = (tr_sc.mean(axis=1), va_sc.mean(axis=1))
    ax.plot(sizes, tr_sc.mean(axis=1), "o-", lw=2, label="training accuracy")
    ax.plot(sizes, va_sc.mean(axis=1), "o-", lw=2, label="validation accuracy (5-fold)")
    ax.fill_between(sizes, va_sc.mean(axis=1) - va_sc.std(axis=1), va_sc.mean(axis=1) + va_sc.std(axis=1), alpha=0.2, color=ORANGE)
    ax.set(title=name, xlabel="training examples", ylim=(0.45, 1.02))
axes[0].set_ylabel("accuracy"); axes[0].legend(loc="lower right"); plt.tight_layout(); plt.show()

(nb_tr, nb_va), (dt_tr, dt_va) = curves.values()
print(f"naive Bayes  : final gap train-val {nb_tr[-1] - nb_va[-1]:.3f}, val gain over last 3 sizes {nb_va[-1] - nb_va[-4]:+.3f}")
print(f"decision tree: final gap train-val {dt_tr[-1] - dt_va[-1]:.3f}, val gain over last 3 sizes {dt_va[-1] - dt_va[-4]:+.3f}")
assert dt_tr[-1] - dt_va[-1] > nb_tr[-1] - nb_va[-1]

How to read them:

- **Naive Bayes:** the two curves meet at a mediocre level and the validation curve is flat. That is *high bias*: more of the same data will not help; you need a better model or better features.
- **Decision tree:** training accuracy is pinned at 100% (memorisation) with a large gap, and validation is still climbing. That is *high variance*: more data should help, and so should regularisation (depth limits) or averaging many trees (a random forest, [notebook 06](06_decision_trees_random_forest_boosting.ipynb)).

The fine-tuning version of this (`tn13`): train on 25%, 50% and 100% of your examples and plot the eval metric. If it is still rising at 100%, more data pays; if it is flat, spend the effort on data *quality* instead.

## 5. Regularisation: keep the capacity, penalise the wiggles

**In plain English:** instead of choosing a smaller model, keep the big one and add a penalty for large weights, so it only uses its flexibility where the data insists.

| | Ridge (L2) | Lasso (L1) |
|---|---|---|
| Penalty | $\alpha\sum_j w_j^2$ | $\alpha\sum_j \lvert w_j\rvert$ |
| Effect on weights | shrinks all of them smoothly, none exactly 0 | shrinks, and sets many *exactly* to 0 |
| Good for | many small, correlated effects; stabilising collinear features | feature selection; when you believe few features matter |
| Closed form? | yes: $(X^\top X + \alpha I)^{-1}X^\top y$ | no: coordinate descent with soft-thresholding |
| Deep-learning cousin | weight decay | (rare) |

First the payoff: the same degree-15 model as above, with a Ridge penalty of α = 0.1. (Choosing α by cross-validation is exercise 1; with only 20 points any CV estimate is noisy, which is itself a lesson.)

In [ ]:
ridge15 = make_pipeline(PolynomialFeatures(15, include_bias=False), StandardScaler(),
                        Ridge(alpha=0.1)).fit(x[:, None], y)
ols15 = poly_model(15).fit(x[:, None], y)
print(f"degree 15, no penalty : val MSE {mse(ols15, x_val, y_val):.3g}")
print(f"degree 15, Ridge(0.1) : val MSE {mse(ridge15, x_val, y_val):.3f}")

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(grid, true_f(grid), color=GREY, ls="--", label="true function")
ax.plot(grid, ols15.predict(grid[:, None]), color=ORANGE, lw=1.5, label="degree 15, no penalty")
ax.plot(grid, ridge15.predict(grid[:, None]), color=AQUA, lw=2.5, label="degree 15 + Ridge")
ax.scatter(x, y, s=18, color=BLUE, zorder=3)
ax.set(title="Same capacity, plus an L2 penalty", xlabel="x", ylabel="y", ylim=(-2, 2)); ax.legend(fontsize=8); plt.show()
assert mse(ridge15, x_val, y_val) < mse(ols15, x_val, y_val) / 5

### Ridge and Lasso from scratch

On `load_diabetes` (10 standardised features). Ridge: centre the data so the intercept is not penalised, then solve one linear system. Lasso: **coordinate descent**. Optimise one weight at a time; the L1 term turns each update into a *soft-threshold*, which snaps small weights to exactly zero:

$$w_j \leftarrow \frac{S(\rho_j, \alpha)}{z_j}, \qquad S(\rho, \alpha) = \text{sign}(\rho)\max(|\rho| - \alpha, 0)$$

where $\rho_j$ is the correlation of feature $j$ with the residual that excludes feature $j$, and $z_j$ is that feature's mean square. scikit-learn's Lasso objective is $\frac{1}{2n}\lVert y - Xw\rVert^2 + \alpha\lVert w\rVert_1$ (note the $\frac{1}{2n}$: α values are not comparable between Ridge and Lasso).

In [ ]:
diab = load_diabetes()
Xd = StandardScaler().fit_transform(diab.data)
yd = diab.target

def ridge_closed_form(X, y, alpha):
    x_mean, y_mean = X.mean(axis=0), y.mean()
    Xc, yc = X - x_mean, y - y_mean
    w = np.linalg.solve(Xc.T @ Xc + alpha * np.eye(X.shape[1]), Xc.T @ yc)
    return w, y_mean - x_mean @ w

w_r, b_r = ridge_closed_form(Xd, yd, alpha=10.0)
sk_r = Ridge(alpha=10.0).fit(Xd, yd)
print("Ridge ours   :", w_r.round(3)); print("Ridge sklearn:", sk_r.coef_.round(3))
assert np.allclose(w_r, sk_r.coef_) and np.isclose(b_r, sk_r.intercept_)

In [ ]:
def lasso_coordinate_descent(X, y, alpha, sweeps=300):
    n_rows, d = X.shape
    x_mean, y_mean = X.mean(axis=0), y.mean()
    Xc, yc = X - x_mean, y - y_mean
    w = np.zeros(d)
    z = (Xc ** 2).mean(axis=0)
    for _ in range(sweeps):
        for j in range(d):
            r_j = yc - Xc @ w + Xc[:, j] * w[j]            # residual without feature j
            rho = Xc[:, j] @ r_j / n_rows
            w[j] = np.sign(rho) * max(abs(rho) - alpha, 0.0) / z[j]    # soft-threshold
    return w, y_mean - x_mean @ w

w_l, b_l = lasso_coordinate_descent(Xd, yd, alpha=2.0)
sk_l = Lasso(alpha=2.0).fit(Xd, yd)
print("Lasso ours   :", w_l.round(3)); print("Lasso sklearn:", sk_l.coef_.round(3))
print(f"weights exactly zero: {int(np.sum(w_l == 0))} of {len(w_l)} -> dropped: {[diab.feature_names[j] for j in np.where(w_l == 0)[0]]}")
assert np.allclose(w_l, sk_l.coef_, atol=1e-3) and np.sum(w_l == 0) >= 2

### Coefficient paths: how each penalty shrinks the weights

Sweep α from tiny to huge and track every coefficient.

In [ ]:
alphas_r = np.logspace(-2, 5, 60)
paths_r = np.array([ridge_closed_form(Xd, yd, a)[0] for a in alphas_r])
alphas_l, paths_l, _ = lasso_path(Xd - Xd.mean(axis=0), yd - yd.mean(), alphas=np.logspace(-2, 1.7, 60))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for j, name in enumerate(diab.feature_names):
    axes[0].plot(alphas_r, paths_r[:, j], lw=1.6)
    axes[1].plot(alphas_l, paths_l[j], lw=1.6, label=name)
axes[0].set(xscale="log", title="Ridge (L2): smooth shrinkage, never exactly 0", xlabel="alpha", ylabel="coefficient")
axes[1].set(xscale="log", title="Lasso (L1): coefficients hit 0 one by one", xlabel="alpha")
for ax in axes:
    ax.axhline(0, color=GREY, lw=0.8)
axes[1].legend(fontsize=7, ncol=2, loc="upper right"); plt.tight_layout(); plt.show()

nonzero = (np.abs(paths_l) > 1e-10).sum(axis=0)
print("non-zero Lasso coefficients from small to large alpha:", nonzero[::-1][::6])
assert nonzero.min() < nonzero.max() and np.all(np.abs(paths_r) > 1e-6)

### Why L1 gives exact zeros: the picture

Minimising loss + penalty is the same as minimising the loss inside a budget region: a circle for L2, a diamond for L1. The solution is where the loss contours first touch the region. The diamond has **corners on the axes**, and a corner is where the contours usually touch first, so one coordinate is exactly zero. The circle has no corners.

In [ ]:
w_ols = np.array([2.0, 0.6])                              # the unpenalised optimum
A = np.array([[1.0, 0.3], [0.3, 0.8]])                    # the loss is a tilted quadratic bowl around it
bowl = lambda W: (((W - w_ols) @ A) * (W - w_ols)).sum(axis=-1)
theta = np.linspace(0, 2 * np.pi, 20_001)
circle = np.c_[np.cos(theta), np.sin(theta)]                        # all w with ||w||_2 = 1
diamond = circle / np.abs(circle).sum(axis=1, keepdims=True)        # all w with ||w||_1 = 1
w_l2, w_l1 = circle[bowl(circle).argmin()], diamond[bowl(diamond).argmin()]

G1, G2 = np.meshgrid(np.linspace(-1.5, 2.6, 200), np.linspace(-1.5, 1.6, 200))
Zb = bowl(np.stack([G1, G2], axis=-1))
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, shape, w_star, name in [(axes[0], circle, w_l2, "L2 budget (circle)"), (axes[1], diamond, w_l1, "L1 budget (diamond)")]:
    ax.contour(G1, G2, Zb, levels=np.geomspace(0.05, 12, 12), cmap="Blues_r", linewidths=0.8)
    ax.fill(shape[:, 0], shape[:, 1], color=ORANGE, alpha=0.25)
    ax.plot(*w_ols, "k*", ms=11, label="unpenalised optimum")
    ax.plot(*w_star, "o", color=ORANGE, ms=9, label=f"penalised optimum ({w_star[0]:.2f}, {w_star[1]:.2f})")
    ax.axhline(0, color=GREY, lw=0.6); ax.axvline(0, color=GREY, lw=0.6)
    ax.set(title=name, xlabel="w1", ylabel="w2", aspect="equal"); ax.legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()
assert abs(w_l1[1]) < 1e-3 and min(abs(w_l2)) > 0.1      # L1 lands on the axis; L2 keeps both weights

## 6. Early stopping (`tn28`)

**In plain English:** an iterative learner (gradient descent, boosting, a neural network) gets more flexible the longer it trains. Watch the validation loss each round and keep the model from the round where it was lowest.

Below, gradient boosting adds one small tree per round (the method itself is [notebook 06](06_decision_trees_random_forest_boosting.ipynb); here it is just "a model that grows every step"). `staged_predict` gives the prediction after every round, so we can draw both curves from a single fit.

In [ ]:
xb = rng.random(200); yb = true_f(xb) + rng.normal(0, 0.3, 200)
xbv = rng.random(2000); ybv = true_f(xbv) + rng.normal(0, 0.3, 2000)
gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.1, max_depth=3, random_state=0).fit(xb[:, None], yb)
tr_curve = [np.mean((p - yb) ** 2) for p in gb.staged_predict(xb[:, None])]
va_curve = [np.mean((p - ybv) ** 2) for p in gb.staged_predict(xbv[:, None])]
best_round = int(np.argmin(va_curve)) + 1

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(range(1, 301), tr_curve, lw=2, label="training loss")
ax.plot(range(1, 301), va_curve, lw=2, label="validation loss")
ax.axvline(best_round, color=GREY, ls="--", label=f"early stop at round {best_round}")
ax.set(title="Training loss keeps falling; validation loss turns around", xlabel="boosting round (think: epoch / step)", ylabel="MSE")
ax.legend(); plt.show()
print(f"validation MSE at round {best_round}: {min(va_curve):.4f}   at round 300: {va_curve[-1]:.4f}")

auto = GradientBoostingRegressor(n_estimators=300, learning_rate=0.1, max_depth=3, n_iter_no_change=10,
                                 validation_fraction=0.2, random_state=0).fit(xb[:, None], yb)
print(f"built-in early stopping (patience 10) kept {auto.n_estimators_} of 300 rounds")
assert va_curve[-1] > 1.2 * min(va_curve) and auto.n_estimators_ < 300

This is the exact picture to describe for LLM fine-tuning (`tn28`): training loss keeps falling, eval loss bottoms out and rises, so keep the checkpoint with the best eval loss (in Hugging Face: `eval_strategy="steps"` plus `load_best_model_at_end=True`), usually after 1 to 3 epochs. Early stopping acts like L2 regularisation: stopping early keeps the weights close to where they started.

## 7. k-fold and stratified k-fold cross-validation

**In plain English:** one validation split is one noisy measurement. **k-fold CV** cuts the data into k parts, trains k times (each time holding out a different part), and averages. Every row is used for validation exactly once. **Stratified** k-fold keeps the class mix the same in every fold.

Sixty samples, sorted by class (30 / 20 / 10), which is how data often arrives:

In [ ]:
y_toy = np.repeat([0, 1, 2], [30, 20, 10])
X_toy = np.zeros((len(y_toy), 1))
class_colors = np.array([BLUE, ORANGE, AQUA])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
fold_counts = {}
for ax, cv, name in [(axes[0], KFold(5), "KFold(5), no shuffle"),
                     (axes[1], StratifiedKFold(5, shuffle=True, random_state=0), "StratifiedKFold(5, shuffle=True)")]:
    M = np.zeros((5, len(y_toy)))
    fold_counts[name] = []
    for i, (_, val_idx) in enumerate(cv.split(X_toy, y_toy)):
        M[i, val_idx] = 1
        fold_counts[name].append(np.bincount(y_toy[val_idx], minlength=3))
    ax.imshow(M, aspect="auto", cmap=ListedColormap(["#cde2fb", "#1c5cab"]), interpolation="nearest")
    ax.scatter(np.arange(len(y_toy)), np.full(len(y_toy), 5.0), c=class_colors[y_toy], marker="s", s=14, clip_on=False)
    ax.set(title=name, xlabel="sample index (bottom strip = class)", yticks=range(5), yticklabels=[f"fold {i + 1}" for i in range(5)])
fig.suptitle("Dark = validation rows in that fold, light = training rows", y=1.02); plt.tight_layout(); plt.show()
for name, counts in fold_counts.items():
    print(f"{name:>34}: validation class counts per fold {[c.tolist() for c in counts]}")
assert all((c == [6, 4, 2]).all() for c in fold_counts["StratifiedKFold(5, shuffle=True)"])

Unshuffled `KFold` on sorted data validates fold 1 on class 0 only, and fold 5 on the rare class only, which trains without it. `StratifiedKFold` gives every fold 6 / 4 / 2. For classification, scikit-learn's `cross_val_score(..., cv=5)` uses stratified folds automatically; pass an explicit splitter when you need shuffling, groups or time order:

| Splitter | Use when |
|---|---|
| `StratifiedKFold` | classification (the default for classifiers) |
| `KFold(shuffle=True)` | regression with independent rows |
| `GroupKFold` | rows are related (same user, patient, document, conversation) |
| `TimeSeriesSplit` | time-ordered data: always validate on the future |
| `RepeatedStratifiedKFold` | small data, and you want a steadier mean and an honest spread |

In [ ]:
X_bc, y_bc = load_breast_cancer(return_X_y=True)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
scores = cross_val_score(clf, X_bc, y_bc, cv=StratifiedKFold(5, shuffle=True, random_state=0))
print("5 fold accuracies:", scores.round(3), f"-> {scores.mean():.3f} ± {scores.std():.3f}")
many = cross_val_score(clf, X_bc, y_bc, cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0))
print(f"50 folds (5 x 10 repeats): mean {many.mean():.3f}, fold scores range {many.min():.3f} to {many.max():.3f}")

Report the **mean and the spread**. A single split on 569 rows could have handed you any number in that range; "model A beat model B by 0.5%" means nothing if the fold-to-fold spread is ±2%.

## 8. `GridSearchCV`: tuning with cross-validation

Split off a test set **first**. Then `GridSearchCV` runs k-fold CV on the training part for every combination, refits the best one on all the training data, and you score the test set once. The model is an RBF support vector machine with two knobs, `C` and `gamma` (explained in [notebook 07](07_knn_svm_naive_bayes.ipynb)); the scaler sits inside the pipeline so each fold scales on its own training rows.

In [ ]:
Xbc_tr, Xbc_te, ybc_tr, ybc_te = train_test_split(X_bc, y_bc, test_size=0.25, stratify=y_bc, random_state=0)
param_grid = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [0.001, 0.01, 0.1, 1]}
search = GridSearchCV(make_pipeline(StandardScaler(), SVC()), param_grid, cv=5).fit(Xbc_tr, ybc_tr)
print("best params:", search.best_params_, f"  mean CV accuracy {search.best_score_:.3f}")
print(f"test accuracy (the one look): {search.score(Xbc_te, ybc_te):.3f}")

heat = search.cv_results_["mean_test_score"].reshape(4, 4)        # rows follow C, columns follow gamma
fig, ax = plt.subplots(figsize=(5.6, 4))
im = ax.imshow(heat, cmap="Blues", vmin=heat.min(), vmax=1.0)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{heat[i, j]:.3f}", ha="center", va="center", fontsize=9, color="white" if heat[i, j] > 0.96 else "black")
ax.set_xticks(range(4), param_grid["svc__gamma"]); ax.set_yticks(range(4), param_grid["svc__C"])
ax.set(title="Mean 5-fold CV accuracy", xlabel="gamma", ylabel="C"); fig.colorbar(im, ax=ax, shrink=0.8); plt.show()
assert np.isclose(heat.max(), search.best_score_)

## 9. Nested cross-validation, and why the test set is touched once

Look at the numbers above: the best mean CV accuracy is higher than the test score. Part of that is ordinary noise, and part is structural: `search.best_score_` is the *best of 16* cross-validated scores, so it carries the same winner's-curse optimism as the 200 useless models in [notebook 01](01_ml_workflow_and_train_test_split.ipynb). Two honest ways to report performance:

1. **A held-out test set** scored once, after all choices are made (section 8).
2. **Nested CV** when data is too small to spare a test set: an *outer* loop estimates performance and an *inner* loop does the tuning, so every outer score comes from rows that played no part in choosing the hyperparameters.

```
outer fold 1:  [ tune on 4/5 with inner 3-fold CV ] -> score on held-out 1/5
outer fold 2:  [ tune on 4/5 with inner 3-fold CV ] -> score on held-out 1/5
...                                                   -> mean = estimate of "tuning + model"
```

In [ ]:
inner = GridSearchCV(make_pipeline(StandardScaler(), SVC()), param_grid, cv=3)
nested = cross_val_score(inner, X_bc, y_bc, cv=StratifiedKFold(3, shuffle=True, random_state=0))
flat = GridSearchCV(make_pipeline(StandardScaler(), SVC()), param_grid, cv=3).fit(X_bc, y_bc)
print(f"non-nested (best_score_ of the grid): {flat.best_score_:.3f}")
print(f"nested CV (outer 3 folds)           : {nested.mean():.3f}  {nested.round(3)}")

On clean data with a small grid the two numbers are close. The gap grows with bigger grids, smaller datasets and noisier labels, which is exactly the prompt-engineering situation: hundreds of prompt variants against 50 cases.

| Set | Used for | How often you look | LLM-world name |
|---|---|---|---|
| Training | fitting weights | constantly | fine-tuning data, few-shot pool |
| Validation / CV folds | choosing models, hyperparameters, thresholds, prompts | as often as you like (it becomes optimistic) | dev set |
| Test | the final, unbiased number | once, at the end | frozen golden eval set (`ev15`) |
| Production | the real exam | continuously | online evals, A/B tests |

`ev26` in one line: offline evals improved and production got worse because the eval set had quietly become a validation set, tuned against until it stopped predicting anything. Freeze the golden set, grow it from real failures, and keep a held-out split you never tune on (`tn20`).

## Try it yourself

**1.** Use `validation_curve` to sweep Ridge's `alpha` for the degree-15 polynomial pipeline on the 20-point data (5-fold CV, scoring `"neg_mean_squared_error"`). Plot train and validation error against alpha. Which side is overfitting?

In [ ]:
# Solution — try it yourself first, then run this
pipe = make_pipeline(PolynomialFeatures(15, include_bias=False), StandardScaler(), Ridge())
alpha_grid = np.logspace(-8, 2, 21)
tr_s, va_s = validation_curve(pipe, x[:, None], y, param_name="ridge__alpha", param_range=alpha_grid,
                              cv=KFold(5, shuffle=True, random_state=0), scoring="neg_mean_squared_error")
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(alpha_grid, -tr_s.mean(axis=1), "o-", label="training MSE")
ax.plot(alpha_grid, -va_s.mean(axis=1), "o-", label="validation MSE (5-fold)")
ax.set(xscale="log", yscale="log", title="Ridge alpha: small = overfit (left), large = underfit (right)",
       xlabel="alpha", ylabel="MSE"); ax.legend(); plt.show()
print(f"best alpha by CV: {alpha_grid[np.argmax(va_s.mean(axis=1))]:.1e}")
# With only 20 points (16 per training fold) CV itself is noisy: the 2,000-point validation set in
# section 5 preferred alpha = 0.1. Small data makes every estimate, including CV, wobbly.

**2.** Write the k-fold loop yourself: for each split, `clone` the model, fit on the training rows, score on the validation rows. Assert your scores equal `cross_val_score` with the same splitter.

In [ ]:
# Solution — try it yourself first, then run this
from sklearn.base import clone
cv = StratifiedKFold(5, shuffle=True, random_state=0)
manual = []
for tr_idx, va_idx in cv.split(X_bc, y_bc):
    fold_model = clone(clf).fit(X_bc[tr_idx], y_bc[tr_idx])
    manual.append(fold_model.score(X_bc[va_idx], y_bc[va_idx]))
print("manual:", np.round(manual, 4))
assert np.allclose(manual, cross_val_score(clf, X_bc, y_bc, cv=cv))

**3.** Print the folds `TimeSeriesSplit(n_splits=4)` makes on 12 time-ordered rows. What guarantee does it give that `KFold` does not?

In [ ]:
# Solution — try it yourself first, then run this
for i, (tr_idx, va_idx) in enumerate(TimeSeriesSplit(n_splits=4).split(np.arange(12))):
    print(f"fold {i + 1}: train {tr_idx.tolist()}  ->  validate {va_idx.tolist()}")
    assert tr_idx.max() < va_idx.min()             # validation is always strictly in the future

## Say it in an interview

- **30-second answer:** "Overfitting is low training error with high validation error: the model has memorised noise. I diagnose it with train-vs-validation curves (against capacity, training-set size or epochs). I fix it with more data, a simpler model, regularisation (L2, L1, early stopping, dropout) or averaging. I pick hyperparameters with cross-validation and keep a test set I look at once, or use nested CV when data is scarce."
- **Bias–variance:** expected error = bias² + variance + noise. High bias: both errors high and close. High variance: a big gap. Measured above: bias² fell by over 100× from degree 1 to the best degree while variance climbed.
- **L1 vs L2:** L2 shrinks smoothly (closed form, weight decay); L1 soft-thresholds weights to exactly zero because the L1 ball has corners on the axes. Elastic net mixes them for correlated groups of features.
- **Learning curves:** a big gap that is still closing means more data will help. Converged-but-low curves mean more data will not; change the model or features.
- **Traps:** tuning on the test set; preprocessing outside the CV loop ([notebook 09](09_feature_engineering_and_pipelines.ipynb)); random k-fold on time series or grouped rows; reporting `best_score_` as the final estimate; forgetting that α means different things in Ridge and Lasso.
- **LLM mapping:** fine-tuning overfits the same way (eval loss rises while train loss falls; keep 1 to 3 epochs and the best checkpoint, `tn28`). A prompt tuned to death on 50 cases is an overfit validation set (`ev26`). The golden set stays frozen (`ev15`).

## Next

- [06 · Decision trees, random forests and boosting](06_decision_trees_random_forest_boosting.ipynb): depth as capacity, bagging to cut variance.
- [09 · Feature engineering and pipelines](09_feature_engineering_and_pipelines.ipynb): cross-validation that cannot leak.
- [Regularisation: dropout and batch norm](../06_deep_learning/03_regularization_dropout_batchnorm.ipynb) and [distillation and catastrophic forgetting](../11_finetuning/07_distillation_and_catastrophic_forgetting.ipynb).
- [A/B testing and statistics for evals](../12_evaluation/05_ab_testing_and_statistics_for_evals.ipynb): is a difference real?
- Theory: [ml_fundamentals course](../../ml_fundamentals/index.html) (chapters 7 and 8), [interview bank](../../ai_interview_prep/index.html).